[Guide Index](README.md)


# 02b Inferencing using RDF, RDFS and OWL 2 RL

Inferencing derives new information from existing graph information according to a set of rules. The semantic W3C specifications provide a variety of methods to perform inferencing as outlined in Section 5 below.

This guide covers RDF, RDFS and OWL 2 RL inferencing using `StarLayerGraph.infer()`.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier sections. Section 3 also needs the `starlayer-reasoning` package (`pip install starlayer-reasoning`).


In [1]:
from starlayer.graph import ENTAILMENT, Namespace, StarLayerGraph

EX = Namespace("http://example.org/")

def show(graph, label="Resulting graph", also_ns=None):
    """Helper function to serialize a graph as Turtle 1.2 - used throughout this
    guide to show a graph's state before/after inferencing.

    Filtered to ex: subjects only. infer() adds a series of RDFS/OWL/XSD
    axiomatic vocabulary triples (rdfs:Resource typing, owl:sameAs reflexivity
    for every built-in class/datatype, etc.) Excluding these triples avoids
    unneeded complexity to the example output.

    also_ns: an additional namespace to include, matched against the predicate or
    object - useful for a triple whose subject isn't ex: but is still worth showing
    (e.g. an owlrl inconsistency triple, whose subject is a synthetic BNode).

    Run print(g.serialize(format="turtle", pretty=True)) to see the complete graph.
    """
    filtered = StarLayerGraph()
    filtered.bind("ex", str(EX))
    filtered.bind("rdf", "http://www.w3.org/1999/02/22-rdf-syntax-ns#")
    filtered.bind("rdfs", "http://www.w3.org/2000/01/rdf-schema#")
    filtered.bind("owl", "http://www.w3.org/2002/07/owl#")
    if also_ns is not None:
        filtered.bind("err", also_ns)
    # terms are pyoxigraph terms: .value is the bare IRI (or label), str() is the N-Triples form
    value = lambda t: getattr(t, "value", "")
    for s, p, o in graph:
        also = also_ns is not None and (value(p).startswith(also_ns) or value(o).startswith(also_ns))
        if value(s).startswith(str(EX)) or also:
            filtered.add((s, p, o))
    print(f"\n{label}")
    print(filtered.serialize(format="turtle", pretty=True, only_used=True))

## 1. How `infer()` works

`StarLayerGraph.infer(entailment=..., mode=...)` has two independent parameters:

- **`entailment`** — controls which ruleset to use in inferencing: any one of `ENTAILMENT.RDF`, `ENTAILMENT.RDFS`, `ENTAILMENT['OWL-RDF-Based']`, or a combination of more than one (e.g. `{ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']}`, or all three at once). (The `ENTAILMENT` IRIs are the W3C regime IRIs; `infer()` also accepts them as `str` or `NamedNode`.)
- **`mode`** — controls what gets returned. The default, `"full"`, is a **new** graph with the original triples plus everything entailed; the original is untouched. Section 4 covers `"delta"` and `"in-place"`.

## 2. RDF and RDFS reasoning

### 2.1 RDF

`entailment=ENTAILMENT.RDF` is the narrowest regime:

- Every predicate used in the graph is typed `rdf:Property` (rdfD2).
- The built-in RDF terms the graph uses are typed as the spec requires, e.g. `rdf:first` as an `rdf:Property` and `rdf:nil` as an `rdf:List`.
- Datatype literal well-typing (rdfD1) is not included.

It can be combined with `ENTAILMENT.RDFS` or `ENTAILMENT['OWL-RDF-Based']`, but in practice rarely adds anything those don't already cover.

In [2]:
g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
        ex:worksAt ex:Acme ;
        ex:skills ( ex:python ex:sql ) .
""", format="turtle")
g.bind("rdf", "http://www.w3.org/1999/02/22-rdf-syntax-ns#")  # so the closure prints rdf: names

closed = g.infer(entailment=ENTAILMENT.RDF)

# show() lists only ex: subjects, so the graphs are printed whole here. Without pretty=True the
# list is written out as rdf:first / rdf:rest triples, and the closure types those terms too
print("\nBefore")
print(g.serialize(format="turtle", bnode_labels="short"))
print("\nAfter infer(entailment=ENTAILMENT.RDF)")
print(closed.serialize(format="turtle", bnode_labels="short"))


Before
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/> .
ex:alice ex:skills _:b0 ;
	ex:worksAt ex:Acme ;
	a ex:Person .
_:b0 rdf:first ex:python ;
	rdf:rest _:b1 .
_:b1 rdf:first ex:sql ;
	rdf:rest rdf:nil .


After infer(entailment=ENTAILMENT.RDF)
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/> .
ex:alice ex:skills _:b0 ;
	ex:worksAt ex:Acme ;
	a ex:Person .
ex:skills a rdf:Property .
ex:worksAt a rdf:Property .
rdf:first a rdf:Property .
rdf:nil a rdf:List .
rdf:rest a rdf:Property .
rdf:type a rdf:Property .
_:b0 rdf:first ex:python ;
	rdf:rest _:b1 .
_:b1 rdf:first ex:sql ;
	rdf:rest rdf:nil .



### 2.2 RDFS

`g.infer(entailment=ENTAILMENT.RDFS)` materializes the RDFS "data" ruleset - four rules:

- **`rdfs:subClassOf`** (rdfs9) - a class's declared superclasses apply to `rdf:type` across any number of hops.
- **`rdfs:domain`** (rdfs2) - a property's declared domain entails a type for its subject.
- **`rdfs:range`** (rdfs3) - a property's declared range entails a type for its object.
- **`rdfs:subPropertyOf`** (rdfs5/rdfs7) - itself reflexive and transitive, so a general relation also matches its more specific sub-relations.

`infer()` also types every term in the data `rdfs:Resource` (rdfs4). It leaves out the RDFS axiomatic triples, datatype typing (rdfD1), and the RDF 1.2 triple-term rules rdfs14 and rdfs14a, which add only vocabulary facts and anonymous nodes.

The example below uses all four rules at once. Alice supervises Bob, and `ex:supervises` declares a domain, a range and a super-property: the domain makes alice a manager, `rdfs:subClassOf` makes her an employee too, the range makes bob an employee, and `rdfs:subPropertyOf` makes alice work with bob.

In [3]:
# alice supervises bob. ex:supervises has a domain, a range and a super-property, and
# Manager is a subclass of Employee, so:
#   rdfs:domain        - alice is a Manager
#   rdfs:subClassOf    - alice is also an Employee
#   rdfs:range         - bob is an Employee
#   rdfs:subPropertyOf - alice worksWith bob
# Every term is also an rdfs:Resource (rdfs4).
g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:supervises rdfs:domain ex:Manager ;
        rdfs:range ex:Employee ;
        rdfs:subPropertyOf ex:worksWith .
    ex:alice ex:supervises ex:bob .
""", format="turtle")

show(g, "Before")

closed = g.infer(entailment=ENTAILMENT.RDFS)

show(closed, "After infer(entailment=ENTAILMENT.RDFS)")
print("original graph untouched:", len(g), "triples")


Before
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:Employee .

ex:alice ex:supervises ex:bob .

ex:supervises rdfs:domain ex:Manager ;
    rdfs:range ex:Employee ;
    rdfs:subPropertyOf ex:worksWith .


After infer(entailment=ENTAILMENT.RDFS)
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource ;
    rdfs:subClassOf ex:Employee .

ex:alice a ex:Employee, ex:Manager, rdfs:Resource ;
    ex:supervises ex:bob ;
    ex:worksWith ex:bob .

ex:bob a ex:Employee, rdfs:Resource .

ex:supervises a rdf:Property, rdfs:Resource ;
    rdfs:domain ex:Manager ;
    rdfs:range ex:Employee ;
    rdfs:subPropertyOf ex:supervises, ex:worksWith .

ex:worksWith a rdf:Property, rdfs:Resource ;
    rdfs:subPropertyOf ex:worksWith .

original graph untouched: 5 triples


## 3. OWL 2 RL reasoning

`g.infer(entailment=ENTAILMENT['OWL-RDF-Based'])` runs a similar forward-chaining closure, but over the OWL 2 RL rule set, producing entailments RDFS alone does not, such as those from `owl:equivalentClass`, `owl:sameAs`, and property characteristics like `owl:TransitiveProperty`.

The OWL-RL regime runs `owlrl`, a Python package that implements RDFS and OWL-RL reasoning on rdflib graphs. rdflib and `owlrl` come with the `starlayer-reasoning` package (`pip install starlayer-reasoning`); `infer()` raises an `ImportError` saying so if the packages are missing.

### 3.1 `owl:equivalentClass`

Two systems label the same role differently — HR calls it `ex:Manager`, the CRM calls the same thing `ex:TeamLead`. Declaring them `owl:equivalentClass` lets an instance of one be recognized as the other.

In [5]:
# alice is a manager, but she is also a teamlead, because manager and teamlead are the same
g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .
    ex:Manager rdfs:subClassOf ex:Employee .

""", format="turtle")

show(g, "Before")

closed = g.infer(entailment=ENTAILMENT['OWL-RDF-Based'])

show(closed, "After infer(entailment=ENTAILMENT['OWL-RDF-Based'])")


Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:Employee ;
    owl:equivalentClass ex:TeamLead .

ex:alice a ex:Manager .


After infer(entailment=ENTAILMENT['OWL-RDF-Based'])
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee owl:sameAs ex:Employee .

ex:Manager rdfs:subClassOf ex:Employee, ex:TeamLead ;
    owl:equivalentClass ex:TeamLead ;
    owl:sameAs ex:Manager .

ex:TeamLead rdfs:subClassOf ex:Employee, ex:Manager ;
    owl:equivalentClass ex:Manager ;
    owl:sameAs ex:TeamLead .

ex:alice a ex:Employee, ex:Manager, ex:TeamLead ;
    owl:sameAs ex:alice .



### 3.2 Property characteristics: `owl:TransitiveProperty`, `owl:SymmetricProperty`, `owl:inverseOf`

Declaring `ex:partOf` to be an `owl:TransitiveProperty` lets a chain of `partOf` facts entail the full transitive closure, not just each direct link. `owl:SymmetricProperty` entails the reverse direction of every asserted fact; `owl:inverseOf` entails a second property's facts from the first's.

In [6]:
# SalesTeam is part of SalesDept, so it is also part of AcmeCorp (transitive).
# bob friendOf alice follows from the symmetric property.
# dave managedBy carol follows from the inverse property.
g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .

    ex:partOf a owl:TransitiveProperty .
    ex:SalesTeam ex:partOf ex:SalesDept .
    ex:SalesDept ex:partOf ex:AcmeCorp .

    ex:friendOf a owl:SymmetricProperty .
    ex:alice ex:friendOf ex:bob .

    ex:manages owl:inverseOf ex:managedBy .
    ex:carol ex:manages ex:dave .
""", format="turtle")

show(g, "Before")

closed = g.infer(entailment=ENTAILMENT['OWL-RDF-Based'])

show(closed, "After infer(entailment=ENTAILMENT['OWL-RDF-Based'])")


Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:SalesDept ex:partOf ex:AcmeCorp .

ex:SalesTeam ex:partOf ex:SalesDept .

ex:alice ex:friendOf ex:bob .

ex:carol ex:manages ex:dave .

ex:friendOf a owl:SymmetricProperty .

ex:manages owl:inverseOf ex:managedBy .

ex:partOf a owl:TransitiveProperty .


After infer(entailment=ENTAILMENT['OWL-RDF-Based'])
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:AcmeCorp owl:sameAs ex:AcmeCorp .

ex:SalesDept ex:partOf ex:AcmeCorp ;
    owl:sameAs ex:SalesDept .

ex:SalesTeam ex:partOf ex:AcmeCorp, ex:SalesDept ;
    owl:sameAs ex:SalesTeam .

ex:alice ex:friendOf ex:bob ;
    owl:sameAs ex:alice .

ex:bob ex:friendOf ex:alice ;
    owl:sameAs ex:bob .

ex:carol ex:manages ex:dave ;
    owl:sameAs ex:carol .

ex:dave ex:managedBy ex:carol ;
    owl:sameAs ex:dave .

ex:friendOf a owl:SymmetricProperty ;
    owl:sameAs ex:friendOf .

ex:managedBy owl:sameAs ex:ma

### 3.3 `owl:sameAs` — congruence closure

Alice Smith and Alice Jones are declared to be the same entity (`owl:sameAs`) - every fact known about one is entailed for the other.

In [7]:
# Alice Smith works at Acme, because Alice Smith and Alice Jones are the same entity.
g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:aliceSmith owl:sameAs ex:aliceJones .
    ex:aliceJones ex:worksAt ex:Acme .
""", format="turtle")

show(g, "Before")

closed = g.infer(entailment=ENTAILMENT['OWL-RDF-Based'])

show(closed, "After infer(entailment=ENTAILMENT['OWL-RDF-Based'])")


Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:aliceJones ex:worksAt ex:Acme .

ex:aliceSmith owl:sameAs ex:aliceJones .


After infer(entailment=ENTAILMENT['OWL-RDF-Based'])
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:Acme owl:sameAs ex:Acme .

ex:aliceJones ex:worksAt ex:Acme ;
    owl:sameAs ex:aliceJones, ex:aliceSmith .

ex:aliceSmith ex:worksAt ex:Acme ;
    owl:sameAs ex:aliceJones, ex:aliceSmith .

ex:worksAt owl:sameAs ex:worksAt .



### 3.4 Running RDFS and OWL 2 RL together

`g.infer(entailment={ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']})` combines both rule sets into a single closure.

What `{ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']}` mostly adds on top of `ENTAILMENT['OWL-RDF-Based']` alone is RDFS's rdfs4 rule, which makes every term an `rdfs:Resource`, and with it `owl:Thing`: both are visible below as `a owl:Thing, rdfs:Resource` on every `ex:` subject. These entailments are absent from an `entailment=ENTAILMENT['OWL-RDF-Based']`-only closure of the same data.

In [8]:
g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .

    ex:Manager rdfs:subClassOf ex:Employee .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .

    ex:partOf a owl:TransitiveProperty .
    ex:SalesTeam ex:partOf ex:SalesDept .
    ex:SalesDept ex:partOf ex:AcmeCorp .
""", format="turtle")

show(g, "Before")

closed = g.infer(entailment={ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']})

show(closed, "After infer(entailment={ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']})")


Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:Employee ;
    owl:equivalentClass ex:TeamLead .

ex:SalesDept ex:partOf ex:AcmeCorp .

ex:SalesTeam ex:partOf ex:SalesDept .

ex:alice a ex:Manager .

ex:partOf a owl:TransitiveProperty .




After infer(entailment={ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']})
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:AcmeCorp a rdfs:Resource, owl:Thing ;
    owl:sameAs ex:AcmeCorp .

ex:Employee a rdfs:Resource, owl:Thing ;
    owl:sameAs ex:Employee .

ex:Manager a rdfs:Resource, owl:Thing ;
    rdfs:subClassOf ex:Employee, ex:Manager, ex:TeamLead ;
    owl:equivalentClass ex:Manager, ex:TeamLead ;
    owl:sameAs ex:Manager .

ex:SalesDept a rdfs:Resource, owl:Thing ;
    ex:partOf ex:AcmeCorp ;
    owl:sameAs ex:SalesDept .

ex:SalesTeam a rdfs:Resource, owl:Thing ;
    ex:partOf ex:AcmeCorp, ex:SalesDept ;
    owl:sameAs ex:SalesTeam .

ex:TeamLead a rdfs:Resource, owl:Thing ;
    rdfs:subClassOf ex:Employee, ex:Manager, ex:TeamLead ;
    owl:equivalentClass ex:Manager, ex:TeamLead ;
    owl:sameAs ex:TeamLead .

ex:alice a

### 3.5 Detecting inconsistencies

`owlrl` also checks a number of the OWL 2 RL "consistency" rules while it reasons — e.g. two classes declared `owl:disjointWith` sharing a member, `owl:sameAs` clashing with `owl:differentFrom`, or a cardinality restriction being violated. It does not raise an exception when one is found; instead it adds an ordinary triple into the result graph. Checking for `err:error` triples in the result is how you detect inconsistencies.

Literal-*value* clashes (e.g. an `owl:FunctionalProperty` given two different literal values for the same subject) are not checked by the current `owlrl` (it skips the rules that compare literal values, dt-diff and dt-eq), so that kind of inconsistency passes through silently. OWL 2 DL reasoning does catch it: see the [OWL 2 DL reasoning guide](02c-owl-dl-reasoning.ipynb).

In [9]:
# Dog and Cat are declared disjoint, but fido is asserted to be both - a genuine inconsistency
ERR = "http://www.daml.org/2002/03/agents/agent-ont#"

g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Dog owl:disjointWith ex:Cat .
    ex:fido a ex:Dog, ex:Cat .
""", format="turtle")

closed = g.infer(entailment=ENTAILMENT['OWL-RDF-Based'])

# also_ns=ERR keeps the err:ErrorMessage / err:error triples in show()'s output, even though
# their subject is a synthetic blank node, not an ex: term.
show(closed, "After infer(entailment=ENTAILMENT['OWL-RDF-Based']) with inconsistency.", also_ns=ERR)

# an inconsistency is an err:error triple in the result
print("inconsistent:", any(p.value == ERR + "error" for _, p, _ in closed))


After infer(entailment=ENTAILMENT['OWL-RDF-Based']) with inconsistency.
@prefix err: <http://www.daml.org/2002/03/agents/agent-ont#> .
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:Cat owl:sameAs ex:Cat .

ex:Dog owl:disjointWith ex:Cat ;
    owl:sameAs ex:Dog .

ex:fido a ex:Cat, ex:Dog ;
    owl:sameAs ex:fido .

err:ErrorMessage owl:sameAs err:ErrorMessage .

err:error owl:sameAs err:error .

_:b0 a err:ErrorMessage ;
    err:error "Disjoint classes http://example.org/Dog and http://example.org/Cat have a common individual http://example.org/fido" .

inconsistent: True


## 4. Delta and in-place mutation

`mode` chooses what `infer()` returns. All three modes reach the same closure (the original data plus everything entailed from it), in different graphs:

- **`"full"`** (default) - a **new** graph with the original triples plus everything entailed; the original is untouched. Sections 2 and 3 use it.
- **`"delta"`** - a **new** graph with just the newly-entailed triples (4.1).
- **`"in-place"`** - the newly-entailed triples are added to the original graph (4.2).

### 4.1 Materializing only the delta

`infer(mode="delta")` returns only the *newly*-entailed triples.

In [ ]:
# A small RDFS example
# delta produces the triples that are added

g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")

closure = g.infer(entailment=ENTAILMENT.RDFS, mode="full")
delta = g.infer(entailment=ENTAILMENT.RDFS, mode="delta")

print("original:", len(g), "triples")
print("full     (mode='full', default):  ", len(closure), "triples - includes the originals")
print("delta    (mode='delta'):          ", len(delta), "triples - just what's new")
show(delta, "delta only")

# the original combined with delta reproduces the full graph exactly
print("\noriginal ∪ delta == full closure:", set(closure) == set(g) | set(delta))

original: 2 triples
full     (mode='full', default):   17 triples - includes the originals
delta    (mode='delta'):           15 triples - just what's new

delta only
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource .

ex:alice a ex:Employee, rdfs:Resource .


original ∪ delta == full closure: True


### 4.2 Mutating in place

`mode="in-place"` adds the newly-entailed triples into the original graph.

Note: if you later delete or edit triples, entailed triples created from the original data remain.

In [11]:
g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")

print("before:", len(g), "triples")
g.infer(entailment=ENTAILMENT.RDFS, mode="in-place")
print("after: ", len(g), "triples")
show(g, "g after mutating in place")

# Removing the fact that justified the entailment
# doesn't retract the entailment itself - infer(mode="in-place") only ever adds triples,
# it never re-checks earlier entailments when the graph changes.
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")
alice, Manager, Employee = EX.alice, EX.Manager, EX.Employee
g.remove((alice, RDF.type, Manager))
print("\nafter removing 'alice a Manager':")
print("alice a Manager:", (alice, RDF.type, Manager) in g, "(gone, as expected)")
print("alice a Employee:", (alice, RDF.type, Employee) in g, "(still there - stale)")

before: 2 triples
after:  17 triples

g after mutating in place
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource ;
    rdfs:subClassOf ex:Employee .

ex:alice a ex:Employee, ex:Manager, rdfs:Resource .


after removing 'alice a Manager':
alice a Manager: False (gone, as expected)
alice a Employee: True (still there - stale)


## 5. Which entailment regime to use

The SPARQL 1.2 [Entailment Regimes](https://www.w3.org/TR/sparql12-entailment/) specification defines six regimes beyond simple entailment. StarLayer supports them in part:

| # | Regime | Status | Use it when |
|---|---|---|---|
| 1 | RDF Entailment | ✅ `infer(entailment=ENTAILMENT.RDF)` (section 2.1) | Rarely on its own: it is the narrowest and cheapest to compute. |
| 2 | RDFS Entailment | ✅ `infer(entailment=ENTAILMENT.RDFS)` (section 2.2) | It is the right default whenever your data doesn't need genuine OWL constructs: fast, and it covers the full RDFS "data" ruleset. |
| 3 | D-Entailment (datatype canonicalization) | ❌ Not supported | |
| 4 | OWL 2 RDF-Based Semantics | ✅ OWL 2 RL, the decidable rule-based fragment - `infer(entailment=ENTAILMENT['OWL-RDF-Based'])` (section 3). ❌ The full regime is undecidable in general and not supported | Your graph uses OWL characteristics RDFS can't express (`owl:equivalentClass` / `owl:equivalentProperty`, property characteristics, `owl:sameAs`). |
| 5 | OWL 2 Direct Semantics | ✅ OWL 2 DL reasoning via HermiT (default) or RustDL - see the [OWL 2 DL reasoning guide](02c-owl-dl-reasoning.ipynb) | You need full OWL 2 DL reasoning (disjunction, full class satisfiability). It has its own guide because it needs a separate reasoner (HermiT or RustDL). |
| 6 | RIF Core Entailment | ❌ Not supported | |

Other inferencing in StarLayer:

- applying a regime at query time, with `query(entailment=...)`, without materializing a closure first: the [SPARQL entailment guide](03a-sparql-entailment.ipynb)
- SHACL rules: [SHACL inference rules](04c-shacl-inference-rules.ipynb)
- SPARQL Rules (SRL): the [SRL guide](03b-srl-sparql-rules.ipynb)